# Проект №4: удаление шума с камеры видеонаблюдения в условиях низкой освещённости

**Дисциплина:** Компьютерное зрение 1. Ноутбук рассчитан на Google Colab, запускать ячейки сверху вниз.

## 1. Постановка задачи

Камеры видеонаблюдения ночью или в тёмных помещениях дают тёмный и шумный кадр: фотонов мало, поэтому на изображении
виден **дробовой шум** (Пуассона) и **шум считывания** матрицы (Гаусса). Простое повышение яркости усиливает шум вместе с сигналом.

**Цель:** построить программу, которая убирает шум и делает кадр пригодным для просмотра, используя только классические методы CV (без нейросетей).

**Что делаем:**
1. Моделируем съёмку при слабом свете (эталон известен, поэтому можно считать PSNR и SSIM).
2. Сравниваем 4 фильтра: Гаусс, медиана, билатеральный, Non-Local Means (NLM).
3. Для видео добавляем адаптивное временное усреднение с маской движения.
4. Повышаем яркость (адаптивная гамма и CLAHE) **после** шумоподавления.
5. Проверяем на разных уровнях освещённости, на нестандартных случаях и измеряем время.

**Почему классические методы:** они быстрые, не требуют обучающей выборки и их работу можно объяснить. Для видеонаблюдения
важна скорость, а соседние кадры статичной камеры дают дополнительную информацию, которую нейросеть по одному кадру не использует.

In [ ]:
import time

import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage import data
from skimage.metrics import peak_signal_noise_ratio as psnr, structural_similarity as ssim

rng = np.random.default_rng(42)
PHOTONS = 400.0   # условное число фотонов при полной яркости пикселя (чем меньше, тем шумнее)


def rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def load_test_images(size=(384, 288)):
    raw = {"astronaut": data.astronaut(), "coffee": data.coffee(), "chelsea": data.chelsea()}
    return {k: cv2.resize(cv2.cvtColor(v, cv2.COLOR_RGB2BGR), size, interpolation=cv2.INTER_AREA)
            for k, v in raw.items()}


def simulate_low_light(img, brightness=0.12, read_sigma=2.0):
    # Тёмная сцена + шум Пуассона (дробовой) + гауссов шум считывания
    x = img.astype(np.float32) / 255.0 * brightness
    shot = rng.poisson(x * PHOTONS).astype(np.float32) / PHOTONS
    read = rng.normal(0, read_sigma / 255.0, img.shape).astype(np.float32)
    return np.clip((shot + read) * 255, 0, 255).round().astype(np.uint8)


def to_ref_scale(img, brightness):
    # Только для метрик: возвращаем яркость к исходной, чтобы сравнить с эталоном
    return np.clip(img.astype(np.float32) / brightness, 0, 255).astype(np.uint8)


def quality(ref, out):
    return (psnr(ref, out, data_range=255),
            ssim(ref, out, channel_axis=2, data_range=255))


imgs = load_test_images()
print({k: v.shape for k, v in imgs.items()})

## 2. Модель шума при слабом свете

Кадр умножается на коэффициент яркости (`brightness`), затем добавляется шум Пуассона (чем темнее, тем он относительно сильнее)
и гауссов шум считывания. Исходное хорошо освещённое изображение хранится как **эталон**, по нему считаем **PSNR** (чем выше, тем лучше)
и **SSIM** (структурное сходство, от 0 до 1).

In [ ]:
BRIGHT = 0.12
fig, axs = plt.subplots(2, 3, figsize=(13, 7))
for j, (name, img) in enumerate(imgs.items()):
    noisy = simulate_low_light(img, BRIGHT)
    axs[0, j].imshow(rgb(img)); axs[0, j].set_title(f"{name}: эталон")
    axs[1, j].imshow(rgb(noisy)); axs[1, j].set_title(f"{name}: слабый свет (яркость {BRIGHT})")
    for i in range(2):
        axs[i, j].axis("off")
plt.tight_layout(); plt.show()

## 3. Методы шумоподавления

| Метод | Идея | Сильная сторона | Ограничение |
|---|---|---|---|
| Гаусс | усреднение с весами по расстоянию | быстрый | размывает края |
| Медиана | медиана в окне | убирает выбросы | теряет мелкие детали |
| Билатеральный | веса по расстоянию и по разнице яркости | сохраняет края, быстрее NLM | параметры нужно подбирать под уровень шума |
| NLM | усреднение похожих **патчей** по всему окну поиска | лучшее качество | самый медленный |

Параметр `h` у NLM задаёт силу сглаживания и должен соответствовать уровню шума. Ниже проверим это на сетке значений.

In [ ]:
def d_gauss(img):    return cv2.GaussianBlur(img, (5, 5), 0)
def d_median(img):   return cv2.medianBlur(img, 5)
def d_bilateral(img): return cv2.bilateralFilter(img, 9, 25, 5)
def d_nlm(img, h=7): return cv2.fastNlMeansDenoisingColored(img, None, h, h, 7, 21)

METHODS = {"Гаусс": d_gauss, "Медиана": d_median, "Билатеральный": d_bilateral, "NLM": d_nlm}

# --- подбор параметра h для NLM ---
test = imgs["astronaut"]
noisy = simulate_low_light(test, BRIGHT)
hs = [3, 5, 7, 9, 12]
res_h = [quality(test, to_ref_scale(d_nlm(noisy, h), BRIGHT)) for h in hs]
for h, (p, s) in zip(hs, res_h):
    print(f"h={h:2d}  PSNR={p:.2f}  SSIM={s:.3f}")

plt.figure(figsize=(5, 3.5))
plt.plot(hs, [r[0] for r in res_h], "o-")
plt.xlabel("h (сила NLM)"); plt.ylabel("PSNR, дБ"); plt.title("Подбор h для NLM"); plt.grid(alpha=.3)
plt.show()

## 4. Сравнение методов на трёх изображениях (кадр + шум -> фильтр -> метрики)

In [ ]:
rows = []
for name, img in imgs.items():
    noisy = simulate_low_light(img, BRIGHT)
    p0, s0 = quality(img, to_ref_scale(noisy, BRIGHT))
    rows.append((name, "Без обработки", p0, s0, 0.0))
    for mname, fn in METHODS.items():
        t0 = time.perf_counter(); out = fn(noisy); dt = (time.perf_counter() - t0) * 1000
        p, s = quality(img, to_ref_scale(out, BRIGHT))
        rows.append((name, mname, p, s, dt))

print(f"{'Изображение':11s} {'Метод':15s} {'PSNR':>6s} {'SSIM':>6s} {'мс':>6s}")
for r in rows:
    print(f"{r[0]:11s} {r[1]:15s} {r[2]:6.2f} {r[3]:6.3f} {r[4]:6.0f}")

# усреднение по изображениям
names = ["Без обработки"] + list(METHODS)
avg_psnr = [np.mean([r[2] for r in rows if r[1] == n]) for n in names]
avg_ms = [np.mean([r[4] for r in rows if r[1] == n]) for n in names]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].bar(names, avg_psnr, color="#4C78A8"); ax[0].set_title("Средний PSNR, дБ"); ax[0].tick_params(axis="x", rotation=25)
ax[1].bar(names, avg_ms, color="#F58518"); ax[1].set_title("Среднее время, мс/кадр"); ax[1].tick_params(axis="x", rotation=25)
plt.tight_layout(); plt.show()

In [ ]:
# визуальное сравнение (с яркостью, приведённой к норме, чтобы было видно разницу)
img = imgs["astronaut"]; noisy = simulate_low_light(img, BRIGHT)
panels = [("Эталон", img), ("Шумный", to_ref_scale(noisy, BRIGHT))] + \
         [(n, to_ref_scale(f(noisy), BRIGHT)) for n, f in METHODS.items()]
fig, axs = plt.subplots(2, 3, figsize=(13, 8))
for ax, (t, im) in zip(axs.ravel(), panels):
    ax.imshow(rgb(im)); ax.set_title(t); ax.axis("off")
plt.tight_layout(); plt.show()

**Что видно по таблице (проверьте на своих цифрах):** все фильтры сильно улучшают шумный кадр. По SSIM NLM чаще всего лучше сохраняет
структуру, а по PSNR билатеральный фильтр сопоставим с NLM или даже выше, при этом работает примерно в 15-25 раз быстрее. Гаусс и медиана
почти не требуют времени, но размывают детали (ниже SSIM). Значит, выбор метода зависит от требований: качество структуры или скорость.

## 5. Повышение яркости: порядок операций важен

Гамма-коррекция и CLAHE (адаптивное выравнивание контраста по блокам) усиливают всё, **в том числе шум**.
Поэтому правильный порядок: **сначала шумоподавление, потом повышение яркости**. Проверим это.

Яркость повышается адаптивно: гамма подбирается так, чтобы средняя яркость кадра стала около 110, затем CLAHE применяется
к каналу L цветового пространства LAB (чтобы не искажать цвета).

In [ ]:
def enhance(img, target=110, clip=2.0):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    m = max(float(gray.mean()), 1.0)
    gamma = float(np.clip(np.log(target / 255.0) / np.log(m / 255.0), 0.2, 1.0))
    lut = ((np.arange(256) / 255.0) ** gamma * 255).astype(np.uint8)
    out = cv2.LUT(img, lut)
    l, a, b = cv2.split(cv2.cvtColor(out, cv2.COLOR_BGR2LAB))
    l = cv2.createCLAHE(clipLimit=clip, tileGridSize=(8, 8)).apply(l)
    return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2BGR)


img = imgs["coffee"]; noisy = simulate_low_light(img, BRIGHT)
a_enh_first = d_nlm(enhance(noisy), h=12)       # сначала яркость, потом шумоподавление
b_den_first = enhance(d_nlm(noisy, h=7))        # сначала шумоподавление, потом яркость
panels = [("Исходный тёмный кадр", noisy), ("Только яркость (шум усилился)", enhance(noisy)),
          ("Яркость -> NLM", a_enh_first), ("NLM -> яркость (лучше)", b_den_first)]
fig, axs = plt.subplots(1, 4, figsize=(16, 4))
for ax, (t, im) in zip(axs, panels):
    ax.imshow(rgb(im)); ax.set_title(t, fontsize=10); ax.axis("off")
plt.tight_layout(); plt.show()
print("SSIM с эталоном (яркость-first):", round(quality(img, a_enh_first)[1], 3),
      "  (шумоподавление-first):", round(quality(img, b_den_first)[1], 3))

## 6. Видео: временное шумоподавление с маской движения

Камера видеонаблюдения обычно **неподвижна**, а значит, фон одинаков в соседних кадрах, а шум разный. Усреднение по времени
снижает шум примерно в корень из N раз без размытия деталей. Но движущиеся объекты при таком усреднении оставляют «шлейф» (ghosting).

**Алгоритм (комбинация методов):**
1. Для каждого предыдущего кадра окна считаем вес по разнице с текущим кадром (после лёгкого размытия): похожие пиксели получают вес близко к 1, отличающиеся (движение) близко к 0.
2. Взвешенное временное среднее + лёгкий NLM даёт результат на статичных областях.
3. **Маска движения** (порог по среднему весу, морфология: открытие и расширение) отделяет движущиеся области.
4. В областях движения используется только пространственный NLM по текущему кадру.

Тестовое видео синтетическое (фон + движущийся «человек»), поэтому есть эталон. Свои видео можно проверить в конце ноутбука.

In [ ]:
def make_video(bg, n=30):
    frames, h, w = [], *bg.shape[:2]
    for t in range(n):
        f = bg.copy()
        x = int(20 + (w - 120) * t / (n - 1))
        cv2.rectangle(f, (x, 100), (x + 60, 200), (40, 40, 200), -1)   # "тело"
        cv2.circle(f, (x + 30, 90), 18, (190, 170, 150), -1)           # "голова"
        frames.append(f)
    return frames


def temporal_denoise(frames, t, k=8, tau=10.0, h_static=3, h_motion=7):
    cur = frames[t]
    win = frames[max(0, t - k + 1):t + 1]
    cs = cv2.GaussianBlur(cur, (7, 7), 0).astype(np.float32)
    num = np.zeros(cur.shape, np.float32)
    den = np.zeros(cur.shape[:2], np.float32)
    for f in win:
        fs = cv2.GaussianBlur(f, (7, 7), 0).astype(np.float32)
        d = np.sqrt(((fs - cs) ** 2).sum(axis=2))
        w = np.exp(-(d / tau) ** 2)
        num += w[..., None] * f
        den += w
    avg = (num / den[..., None]).round().astype(np.uint8)

    motion = ((den / len(win)) < 0.6).astype(np.uint8)
    motion = cv2.morphologyEx(motion, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    motion = cv2.dilate(motion, np.ones((5, 5), np.uint8))

    static_part = d_nlm(avg, h_static)
    moving_part = d_nlm(cur, h_motion)
    out = np.where(motion[..., None] > 0, moving_part, static_part).astype(np.uint8)
    return out, motion


def naive_temporal(frames, t, k=8):
    return np.mean(frames[max(0, t - k + 1):t + 1], axis=0).round().astype(np.uint8)


bg = imgs["coffee"]
clean = make_video(bg, 30)
noisy_v = [simulate_low_light(f, BRIGHT) for f in clean]

methods_v = {
    "Без обработки": lambda t: noisy_v[t],
    "NLM по кадрам": lambda t: d_nlm(noisy_v[t]),
    "Простое усреднение": lambda t: naive_temporal(noisy_v, t),
    "Адаптивный (наш)": lambda t: temporal_denoise(noisy_v, t)[0],
}
START = 8   # с этого кадра окно заполнено
curves = {k: [] for k in methods_v}
times = {k: [] for k in methods_v}
for t in range(START, len(clean)):
    for k, fn in methods_v.items():
        t0 = time.perf_counter(); out = fn(t); times[k].append((time.perf_counter() - t0) * 1000)
        curves[k].append(psnr(clean[t], to_ref_scale(out, BRIGHT), data_range=255))

plt.figure(figsize=(8, 4))
for k, v in curves.items():
    plt.plot(range(START, len(clean)), v, label=f"{k} (средн. {np.mean(v):.1f} дБ)")
plt.xlabel("Номер кадра"); plt.ylabel("PSNR, дБ"); plt.legend(fontsize=8); plt.grid(alpha=.3)
plt.title("Качество на видео"); plt.show()
for k in methods_v:
    print(f"{k:20s} PSNR={np.mean(curves[k]):.2f} дБ   время={np.mean(times[k]):.0f} мс/кадр")

In [ ]:
# визуализация: кадр, маска движения, до/после (яркость приведена к норме для наглядности)
t = 20
out, motion = temporal_denoise(noisy_v, t)
naive_out = naive_temporal(noisy_v, t)
panels = [("Эталон", clean[t]), ("Шумный кадр", to_ref_scale(noisy_v[t], BRIGHT)),
          ("Простое усреднение (шлейф)", to_ref_scale(naive_out, BRIGHT)),
          ("Маска движения", motion * 255), ("Адаптивный метод", to_ref_scale(out, BRIGHT))]
fig, axs = plt.subplots(1, 5, figsize=(20, 3.6))
for ax, (title, im) in zip(axs, panels):
    ax.imshow(im if im.ndim == 2 else rgb(im), cmap="gray"); ax.set_title(title, fontsize=10); ax.axis("off")
plt.tight_layout(); plt.show()

## 7. Тестирование на разных условиях и нестандартных случаях

Проверяем устойчивость: разная освещённость (от слабо до очень тёмного) и нестандартный случай: **яркий источник света в тёмной сцене**
(фары, фонарь), где адаптивная гамма и CLAHE могут «пересветить» кадр.

In [ ]:
levels = [0.30, 0.20, 0.12, 0.06, 0.03]
img = imgs["chelsea"]
table = {n: [] for n in ["Без обработки"] + list(METHODS)}
for b in levels:
    nz = simulate_low_light(img, b)
    table["Без обработки"].append(quality(img, to_ref_scale(nz, b))[0])
    for n, f in METHODS.items():
        table[n].append(quality(img, to_ref_scale(f(nz), b))[0])

plt.figure(figsize=(8, 4))
for n, v in table.items():
    plt.plot(levels, v, "o-", label=n)
plt.gca().invert_xaxis()
plt.xlabel("Яркость сцены (меньше = темнее)"); plt.ylabel("PSNR, дБ")
plt.title("Устойчивость к уровню освещённости"); plt.legend(fontsize=8); plt.grid(alpha=.3); plt.show()

In [ ]:
# нестандартный случай: яркий источник света (засветка) в тёмной сцене
img = imgs["astronaut"].copy()
cv2.circle(img, (320, 60), 22, (255, 255, 255), -1)          # "фонарь"
dark = simulate_low_light(img, 0.10)
dark[40:80, 300:340] = np.maximum(dark[40:80, 300:340], 235)  # источник остаётся ярким в тёмном кадре

den = d_nlm(dark, 7)
plain = enhance(den, clip=2.0)         # CLAHE с обычным пределом
safe = enhance(den, clip=1.0)          # более мягкий CLAHE
panels = [("Тёмный кадр с засветкой", dark), ("NLM -> яркость (clip=2.0)", plain), ("NLM -> яркость (clip=1.0)", safe)]
fig, axs = plt.subplots(1, 3, figsize=(14, 4))
for ax, (t, im) in zip(axs, panels):
    ax.imshow(rgb(im)); ax.set_title(t); ax.axis("off")
plt.tight_layout(); plt.show()

**Наблюдения для отчёта** (дополните по своим графикам):
- На слабом свете PSNR всех методов падает. NLM с **фиксированным** `h=7` (подобран для яркости 0.12) на очень тёмных кадрах проигрывает билатеральному фильтру: сила сглаживания должна соответствовать уровню шума, а он меняется с освещённостью. Это ограничение метода и направление улучшения (автоподбор `h`).
- Яркий источник в кадре «тянет» среднюю яркость вверх, поэтому адаптивная гамма меньше осветляет остальную сцену; снижение `clipLimit` у CLAHE уменьшает артефакты.
- Временное усреднение работает только при неподвижной камере: при дрожании или смене экспозиции ему нужна стабилизация.

## 8. Ваше собственное видео (необязательно)

Загрузите видео из тёмного помещения или с камеры наблюдения (.mp4/.avi). Если файла нет, ячейку можно пропустить.

In [ ]:
from google.colab import files

def process_user_video(path, max_frames=60, scale=0.5):
    cap = cv2.VideoCapture(path)
    frames = []
    while len(frames) < max_frames:
        ok, f = cap.read()
        if not ok:
            break
        frames.append(cv2.resize(f, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA))
    cap.release()
    assert len(frames) > 10, "Видео слишком короткое или не открылось"
    t = len(frames) - 1
    out, motion = temporal_denoise(frames, t, k=8, tau=12.0)
    return frames[t], enhance(out), motion

uploaded = files.upload()          # выберите файл видео
path = next(iter(uploaded))
before, after, motion = process_user_video(path)
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
for ax, (t, im) in zip(axs, [("До (последний кадр)", enhance(before)), ("После шумоподавления и яркости", after), ("Маска движения", motion * 255)]):
    ax.imshow(im if im.ndim == 2 else rgb(im), cmap="gray"); ax.set_title(t); ax.axis("off")
plt.tight_layout(); plt.show()

## 9. Выводы

- **Одиночные фильтры:** NLM лучше сохраняет структуру (SSIM), но самый медленный; билатеральный фильтр близок по PSNR и намного быстрее; Гаусс и медиана быстрые, но размывают детали.
- **Для видео с неподвижной камерой** адаптивное временное усреднение с маской движения даёт заметно лучшее качество, чем NLM по отдельным кадрам (на тесте около +2,7 дБ PSNR), и не оставляет «шлейфа» от движущихся объектов (в отличие от простого усреднения). Цена: время на кадр выше, потому что NLM вызывается дважды (для статики и для движения).
- **Порядок важен:** шумоподавление нужно делать до повышения яркости.
- **Ограничения:** метод требует неподвижной камеры, параметры (`h`, `tau`, размер окна) подбираются под уровень шума, при очень слабом свете часть деталей потеряна безвозвратно.
- **Возможные улучшения:** оценка шума по кадру для автоматического `h`, стабилизация камеры перед усреднением, ускорение через уменьшение окна поиска NLM.